# OpenAI API Key Validation

This notebook verifies that your OpenAI API key is working by sending a simple authenticated request and checking the response.


In [ ]:
import os
import json
import requests
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

# Load .env manually if present.
env_path = Path.cwd() / ".env"
if env_path.exists():
    with env_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key and value and key not in os.environ:
                os.environ[key] = value

api_key = os.environ.get("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError(
        "OPENROUTER_API_KEY is not set. Ensure .env contains OPENROUTER_API_KEY or set it in the notebook environment.\n"
        "Example .env entry: OPENROUTER_API_KEY=sk-..."
    )

print("OpenRouter API key loaded.")

url = "https://openrouter.ai/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}
model = "openai/gpt-oss-20b:free"

output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="10px", width="100%", max_height="300px", overflow_y="auto"))
user_input = widgets.Text(placeholder="Type your prompt here...", description="Prompt:", layout=widgets.Layout(width="100%"))
send_button = widgets.Button(description="Send", button_style="primary")


def send_message(_):
    prompt = user_input.value.strip()
    if not prompt:
        return

    with output:
        clear_output()
        print("Sending request...")

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 120,
        "temperature": 0.7,
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60, allow_redirects=True)
        with output:
            clear_output()
            print("Status:", response.status_code)
            print("Content-Type:", response.headers.get("Content-Type"))
            try:
                data = response.json()
            except ValueError:
                print("Response is not valid JSON:")
                print(response.text)
                return

            if response.status_code == 200:
                choice = data.get("choices", [])[0] if data.get("choices") else None
                if choice and choice.get("message"):
                    print("You:", prompt)
                    print("Assistant:", choice["message"]["content"].strip())
                else:
                    print("Unexpected response format:")
                    print(json.dumps(data, indent=2))
            else:
                print("API returned an error:")
                print(json.dumps(data, indent=2))
    except Exception as exc:
        with output:
            clear_output()
            print("Request failed:", repr(exc))


send_button.on_click(send_message)

display(widgets.VBox([user_input, send_button, output]))


OpenRouter API key loaded.
